# Popularity-bias RecSys on Colab

Runs the Amazon Fashion pipeline end to end: clone repo, download raw data, extract to parquet, run Item-CF on the GPU.

**Before you start:** Runtime > Change runtime type > T4 GPU.

Amazon Fashion is the default because it is small enough for Colab's ~12 GB RAM / ~100 GB disk. Clothing_Shoes_and_Jewelry (~66M reviews, ~9 GB parquet) will not fit in free-tier RAM.

## 1. Clone the repo

In [ ]:
import os

# The scripts expect raw files in the folder ABOVE the repo, so we clone into /content/work/.
# Repo is cloned to /content/work/popularity-bias-recsys, raw data goes in /content/work/.
os.makedirs('/content/work', exist_ok=True)
%cd /content/work
if not os.path.exists('popularity-bias-recsys'):
    !git clone https://github.com/chenshiyao-del/popularity-bias-recsys.git
%cd /content/work/popularity-bias-recsys
!git pull
!ls Models

If `item_cf_torch.py` is missing from the listing above, it hasn't been pushed to GitHub yet. Push it (see Workflow.md) or upload it with the Colab file browser into `Models/`.

## 2. Check the GPU and libraries

In [ ]:
import torch, pandas, pyarrow, scipy, sklearn
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
print('pandas', pandas.__version__, '| pyarrow', pyarrow.__version__)
!free -h | head -2
!df -h /content | tail -1

## 3. (Optional) Mount Google Drive to cache the data

Colab wipes `/content` when the runtime resets. Mounting Drive lets you keep the downloaded files and parquet outputs so you don't redo steps 4-5. Skip this cell if you don't care.

In [ ]:
USE_DRIVE = False   # set True to cache raw + parquet on Drive

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE = '/content/drive/MyDrive/popularity_bias_cache'
    os.makedirs(CACHE, exist_ok=True)
    print('Cache dir:', CACHE)

## 4. Download the raw data

From the Amazon Reviews 2023 dataset on Hugging Face (https://amazon-reviews-2023.github.io/). Files go in `/content/work/`, where the extract scripts look for them.

In [ ]:
CATEGORY = 'Amazon_Fashion'
BASE = 'https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw'

%cd /content/work
if USE_DRIVE and os.path.exists(f'{CACHE}/{CATEGORY}.jsonl'):
    !cp {CACHE}/{CATEGORY}.jsonl {CACHE}/meta_{CATEGORY}.jsonl /content/work/
else:
    !wget -q --show-progress -O {CATEGORY}.jsonl {BASE}/review_categories/{CATEGORY}.jsonl
    !wget -q --show-progress -O meta_{CATEGORY}.jsonl {BASE}/meta_categories/meta_{CATEGORY}.jsonl
    if USE_DRIVE:
        !cp {CATEGORY}.jsonl meta_{CATEGORY}.jsonl {CACHE}/
!ls -lh /content/work

## 5. Extract reviews + metadata to parquet

In [ ]:
%cd /content/work/popularity-bias-recsys
# The HF_HOME lines in these scripts point at D:\ and are harmless on Linux.
!python "Amazon Fashion/extract_review_amazon_fashion.py"
!python "Amazon Fashion/extract_meta_amazon_fashion.py"
!ls -lh data/interim

## 6. Run Item-CF on the GPU

`--sample-users 0` uses every user. Drop it back to `20000` if you hit out-of-memory. Use `--gpu-mem-gb` to control the dense chunk size (T4 has ~15 GB, so `4` is safe).

In [ ]:
%cd /content/work/popularity-bias-recsys
!python Models/item_cf_torch.py data/interim/Amazon_Fashion_full.parquet \
    --meta data/interim/Amazon_Fashion_meta.parquet \
    --k 10 --sample-users 0 --gpu-mem-gb 4

## 7. (Optional) Save parquet outputs to Drive

In [ ]:
if USE_DRIVE:
    !cp data/interim/*.parquet {CACHE}/
    print('Saved to', CACHE)